In [2]:
import pandas as pd
import numpy as np

In [3]:
df_framingham = pd.read_csv('data/24-framingham.csv')
df_framingham.head()

,Sex,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,male,39,4.0,No,0.0,0.0,0,0,No,195.0,106.0,70.0,26.97,80.0,77.0,0
1,female,46,2.0,No,0.0,0.0,0,0,No,250.0,121.0,81.0,28.73,95.0,76.0,0
2,male,48,1.0,Yes,20.0,0.0,0,0,No,245.0,127.5,80.0,25.34,75.0,70.0,0
3,female,61,3.0,Yes,30.0,0.0,0,1,No,225.0,150.0,95.0,28.58,65.0,103.0,1
4,female,46,3.0,Yes,23.0,0.0,0,0,No,285.0,130.0,84.0,23.10,85.0,85.0,0


In [4]:
df_stroke = pd.read_csv('data/26-stroke.csv')
df_stroke.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [22]:
df_stroke['smoking_status'].value_counts()

smoking_status
0.0    2777
1.0     789
Name: count, dtype: int64

In [5]:
df_framingham["gender"]            = df_framingham["Sex"].str.capitalize()
df_framingham["hypertension"]      = df_framingham["prevalentHyp"].astype(int)
df_framingham["heart_disease"]     = df_framingham["TenYearCHD"].astype(int)
df_framingham["stroke"]            = df_framingham["prevalentStroke"].astype(int)
df_framingham["avg_glucose_level"] = df_framingham["glucose"]
df_framingham["smoking_status"] = df_framingham["currentSmoker"].map({"Yes": 1, "No": 0})

df_framingham["ever_married"]   = np.nan
df_framingham["work_type"]      = np.nan
df_framingham["Residence_type"] = np.nan
df_stroke["smoking_status"] = df_stroke["smoking_status"].map({
    "smokes":          1,
    "formerly smoked": 0,
    "never smoked":    0,
    "Unknown":         np.nan
})

bonus_cols = ["education", "cigsPerDay", "BPMeds", "totChol",
              "sysBP", "diaBP", "heartRate"]

df_framingham_aligned = df_framingham[[
    "gender", "age", "hypertension", "heart_disease",
    "ever_married", "work_type", "Residence_type",
    "avg_glucose_level", "BMI", "smoking_status", "stroke",
    *bonus_cols
]].rename(columns={"BMI": "bmi"})

df_stroke["source"]               = "stroke"
df_framingham_aligned["source"]   = "framingham"

for col in bonus_cols:
    df_stroke[col] = np.nan

df_stroke_aligned = df_stroke[[
    "gender", "age", "hypertension", "heart_disease",
    "ever_married", "work_type", "Residence_type",
    "avg_glucose_level", "bmi", "smoking_status", "stroke",
    *bonus_cols, "source"
]]

merged = pd.concat([df_stroke_aligned, df_framingham_aligned], ignore_index=True)
merged["hypertension"]    = merged["hypertension"].astype("Int64")
merged["heart_disease"]   = merged["heart_disease"].astype("Int64")
merged["stroke"]          = merged["stroke"].astype("Int64")
merged["smoking_status"]  = merged["smoking_status"].astype("Int64")
merged["age"]             = merged["age"].astype(float)


In [7]:
for col in ["bmi", "avg_glucose_level", "cigsPerDay",
            "totChol", "sysBP", "diaBP", "heartRate"]:
    merged[col] = merged.groupby("source")[col].transform(
        lambda x: x.fillna(x.median())
    )

for col in ["smoking_status", "BPMeds", "education",
            "ever_married", "work_type", "Residence_type"]:
    merged[col] = merged.groupby("source")[col].transform(
        lambda x: x.fillna(x.mode()[0]) if x.notna().any() else x
    )

In [19]:
nan_columns = ['ever_married', 'work_type', 'Residence_type', 'smoking_status', 'stroke', 'education', 'cigsPerDay', 'BPMeds', 'totChol', 'sysBP', 'diaBP', 'heartRate']

In [42]:
merged.count()

gender               9350
age                  9350
hypertension         9350
heart_disease        9350
ever_married         5110
work_type            5110
Residence_type       5110
avg_glucose_level    9350
bmi                  9350
smoking_status       4240
stroke               9350
education            4240
cigsPerDay           4240
BPMeds               4240
totChol              4240
sysBP                4240
diaBP                4240
heartRate            4240
source               9350
dtype: int64